In [5]:
from langchain_neo4j import Neo4jGraph
from configuration.config import *

graph = Neo4jGraph(
    url=NEO4J_CONFIG["uri"],
    username=NEO4J_CONFIG["auth"][0],
    password=NEO4J_CONFIG["auth"][1],
)

In [6]:
print(graph.schema)

Node properties:
Category1 {id: INTEGER, name: STRING}
Category2 {id: INTEGER, name: STRING}
Category3 {id: INTEGER, name: STRING}
BaseAttrName {id: INTEGER, name: STRING}
BaseAttrValue {id: INTEGER, name: STRING}
SPU {id: INTEGER, name: STRING}
SKU {id: INTEGER, name: STRING}
Trademark {id: INTEGER, name: STRING}
SaleAttrName {id: INTEGER, name: STRING}
SaleAttrValue {id: INTEGER, name: STRING}
Tag {id: STRING, name: STRING}
Relationship properties:

The relationships:
(:Category1)-[:Have]->(:BaseAttrName)
(:Category2)-[:Belong]->(:Category1)
(:Category2)-[:Have]->(:BaseAttrName)
(:Category3)-[:Have]->(:BaseAttrName)
(:Category3)-[:Belong]->(:Category2)
(:BaseAttrName)-[:Have]->(:BaseAttrValue)
(:SPU)-[:Have]->(:Tag)
(:SPU)-[:Have]->(:SaleAttrName)
(:SPU)-[:Belong]->(:Trademark)
(:SPU)-[:Belong]->(:Category3)
(:SKU)-[:Have]->(:BaseAttrValue)
(:SKU)-[:Have]->(:SaleAttrValue)
(:SKU)-[:Belong]->(:SPU)
(:SaleAttrName)-[:Have]->(:SaleAttrValue)


In [7]:
# 查询
cypher = "MATCH (n) RETURN n LIMIT 5"
graph.query(cypher)

[{'n': {'name': '图书、音像、电子书刊', 'id': 1}},
 {'n': {'name': '手机', 'id': 2}},
 {'n': {'name': '家用电器', 'id': 3}},
 {'n': {'name': '数码', 'id': 4}},
 {'n': {'name': '家居家装', 'id': 5}}]

In [8]:
from langchain_openai import ChatOpenAI
import dotenv

dotenv.load_dotenv()
# 定义大模型，同时temperature=0是选取概率最大的词来生成；如果等于1，则是按照概率随机生成。
llm = ChatOpenAI(
    model="gpt-4o-mini", temperature=0
)


In [9]:
from langchain_neo4j import GraphCypherQAChain

# 定义chain
chain = GraphCypherQAChain.from_llm(graph=graph, llm=llm, verbose=True, allow_dangerous_requests=True)

In [13]:
result = chain.invoke({"query": "苹果有哪些产品？"})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (spu:SPU)-[:Belong]->(trademark:Trademark {name: '苹果'}) RETURN spu.name
Full Context:
[{'spu.name': 'Apple iPhone 12'}, {'spu.name': 'Apple iPhone 16 Pro'}]

> Finished chain.


In [14]:
#《apple和苹果》这种问题需要通过实体对齐解决
result = chain.invoke({"query": "apple有哪些产品？"})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (t:Trademark {name: 'apple'})<-[:Belong]-(spu:SPU) RETURN spu.name
Full Context:
[]

> Finished chain.
